# Advanced Retrieval Pipeline with Metadata Filtering and Query Refinement

This notebook implements a retrieval system that:
1.  Analyzes user queries to extract metadata filters (Year, Dept, etc.).
2.  Refines the query for better semantic matching.
3.  Performs a filtered similarity search in ChromaDB.

In [64]:
import os
import json
from typing import List, Optional
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# Load environment variables
load_dotenv("../.env")

# Configuration
DB_DIR = "../metadata_n_db/chroma_db"

if not os.path.exists(DB_DIR):
    print(f"Warning: DB directory {DB_DIR} does not exist. Please check the path.")
else:
    print(f"DB Directory: {os.path.abspath(DB_DIR)}")

DB Directory: /home/rishabh/coding/pro/RAG/metadata_n_db/chroma_db


In [71]:
class RAGRetriever:
    def __init__(self, db_dir: str, api_key_env: str = "GEMINI1", model_name: str = "gemini-1.5-flash"):
        """
        Initialize the RAG Retriever with Embeddings, Vector Store, and LLM.
        """
        self.api_key = os.getenv(api_key_env)
        if not self.api_key:
            raise ValueError(f"API Key environment variable '{api_key_env}' not found.")
            
        print(f"Initializing RAGRetriever with model: {model_name}")
        
        # 1. Initialize LLM
        self.llm = ChatGoogleGenerativeAI(
            model=model_name,
            temperature=0,
            google_api_key=self.api_key
        )
        
        # 2. Initialize Embeddings
        self.embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            google_api_key=self.api_key
        )
        
        # 3. Load Vector Store
        self.vectorstore = Chroma(
            persist_directory=db_dir,
            embedding_function=self.embeddings
        )
        print(f"Vector Store loaded with {self.vectorstore._collection.count()} documents.")
        
        # 4. Setup Chains
        self._setup_reranking_chain()
        
    def _setup_reranking_chain(self):
        """Sets up the LLM chain used for reranking documents."""
        
        # Output Schema
        class RelevanceScore(BaseModel):
            index: int = Field(description="The index of the document in the provided list")
            relevance_score: float = Field(description="A score from 0.0 to 1.0 indicating relevance")
            reasoning: str = Field(description="Brief reason why this document matches the constraints")

        class RankedDocuments(BaseModel):
            ranked_results: List[RelevanceScore]

        self.rerank_parser = JsonOutputParser(pydantic_object=RankedDocuments)

        # Prompt
        self.rerank_prompt = PromptTemplate(
            template="""You are an expert relevance ranker. 
            The user asked: "{query}"
            
            Below is a list of document snippets retrieved from a database. 
            Your job is to evaluate each snippet and determine if it truly answers the user's specific constraints (e.g., specific year, specific department, specific format).
            
            If a document is relevant, assign a high score (0.7 - 1.0).
            If it is topic-adjacent but misses the specific constraint (e.g., wrong year), assign a low score (0.0 - 0.3).
            
            Documents:
            {doc_list}
            
            Return the output as valid JSON matching the format instructions.
            {format_instructions}
            """,
            input_variables=["query", "doc_list"],
            partial_variables={"format_instructions": self.rerank_parser.get_format_instructions()},
        )

        self.rerank_chain = self.rerank_prompt | self.llm | self.rerank_parser

    def retrieve(self, query: str, k_fetch: int = 10, top_n: int = 3) -> List:
        """
        Retrieves documents using a two-stage process:
        1. Wide Retrieval: Fetch k_fetch documents using vector similarity.
        2. Reranking: Use LLM to score and filter the top_n documents.
        """
        print(f"--- 1. Wide Retrieval for: '{query}' ---")
        
        # 1. Fetch more documents than needed (High Recall)
        initial_docs = self.vectorstore.similarity_search(query, k=k_fetch)
        
        print(f"\n[Log] Initial {len(initial_docs)} Documents Retrieved:")
        for i, doc in enumerate(initial_docs):
            print(f"  [{i}] Source: {doc.metadata.get('source')} | Page: {doc.metadata.get('page')} | Title: {doc.metadata.get('title')}")
        
        # 2. Format for the LLM
        doc_texts = []
        for i, doc in enumerate(initial_docs):
            snippet = f"Doc ID {i}:\nMetadata: {doc.metadata}\nContent: {doc.page_content[:400]}..." 
            doc_texts.append(snippet)
        
        combined_text = "\n\n".join(doc_texts)

        print(f"\n--- 2. Reranking {len(initial_docs)} documents ---")
        
        try:
            # 3. Call the LLM Judge
            ranking_result = self.rerank_chain.invoke({"query": query, "doc_list": combined_text})
            
            # 4. Sort and Filter
            sorted_ranks = sorted(ranking_result['ranked_results'], key=lambda x: x['relevance_score'], reverse=True)
            
            final_docs = []
            print("\n--- Top Selected Documents ---")
            for item in sorted_ranks[:top_n]:
                if item['relevance_score'] < 0.5:
                    print(f"Skipping Doc {item['index']} (Low Score: {item['relevance_score']})")
                    continue
                    
                original_doc = initial_docs[item['index']]
                print(f"Score: {item['relevance_score']} | Doc Source: {original_doc.metadata.get('source')}")
                print(f"Reasoning: {item['reasoning']}")
                final_docs.append(original_doc)
                
            return final_docs

        except Exception as e:
            print(f"Reranking failed: {e}. Falling back to raw vector search.")
            return initial_docs[:top_n]

In [72]:
# Initialize the Retriever
# You can switch models here (e.g., "gemini-2.5-flash-lite" if available)
retriever = RAGRetriever(
    db_dir=DB_DIR, 
    api_key_env="GEMINI2", 
    model_name="gemini-2.5-flash-lite"
)

Initializing RAGRetriever with model: gemini-2.5-flash-lite
Vector Store loaded with 2339 documents.


In [73]:
# Test 2: Syllabus
retriever.retrieve("Syllabus of Mathematics-I for first year")

--- 1. Wide Retrieval for: 'Syllabus of Mathematics-I for first year' ---

[Log] Initial 10 Documents Retrieved:
  [0] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Page: 4 | Title: Curricular Structure for B.Tech. I Year
  [1] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Page: 4 | Title: Curricular Structure for B.Tech. I Year
  [2] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Page: 5 | Title: Curricular Structure for B.Tech. I Year
  [3] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Page: 0 | Title: Curricular Structure for B.Tech. I Year
  [4] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Page: 9 | Title: Curricular Structure for B.Tech. I Year
  [5] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Page: 4 | Title: Curricular Structure for B.Tech. I Year
  [6] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Page: 1 | Title: Curricular Structure for B.Tech. I Year
  [7] Source: ../pdfs/18thScroll_2024.pdf | Page: 23 | Title: Scroll of Awardees
  [8] Source: ../pdfs/RM-II_Syllabus

[Document(id='6629a86c-ce04-4b99-86a9-8fc10173fb40', metadata={'Dept': 'Institute', 'moddate': '2016-04-19T09:50:49+05:30', 'page_label': '5', 'creator': 'Microsoft® Word 2013', 'source': '../pdfs/1st_Year_Scheme_Syallbus.pdf', 'audience': 'UG', 'producer': 'Microsoft® Word 2013', 'summary': 'This document outlines the curricular structure for the first year of the B.Tech. program, common to all branches at MNIT Jaipur.', 'author': 'Valued Customer', 'total_pages': 15, 'title': 'Curricular Structure for B.Tech. I Year', 'creationdate': '2016-04-19T09:50:49+05:30', 'page': 4, 'year': 'Unknown', 'doc_type': 'Syllabus'}, page_content='Differential Calculus :  Curvature , Concavity, convexity and points of  Inflexion, \nAsymptotes, Partial differentiation, Euler’s theorem on homogeneous functions, Total \ndifferentiation, Approximate calculation, Curve tracing (Cartesian and five polar curves - \nFolium of Descartes, Limacon, Cardioids, Lemniscates of Bernoulli and Equiangular \nspiral). \

In [74]:
# Test 3: Fee Structure (Specific Year)
retriever.retrieve("Fee Structure year 2016 admitted students for Btech students")

--- 1. Wide Retrieval for: 'Fee Structure year 2016 admitted students for Btech students' ---

[Log] Initial 10 Documents Retrieved:
  [0] Source: ../pdfs/Fee_Structure_UG_2016-17.pdf | Page: 0 | Title: Fee Structure for B. Tech. /B.Arch.
  [1] Source: ../pdfs/Fee_Structure_UG_2016-17.pdf | Page: 2 | Title: Fee Structure for B. Tech. /B.Arch.
  [2] Source: ../pdfs/FeeUG2025-26.pdf | Page: 0 | Title: Fee Structure for B. Tech.
  [3] Source: ../pdfs/Fee_Structure_UG_2016-17.pdf | Page: 1 | Title: Fee Structure for B. Tech. /B.Arch.
  [4] Source: ../pdfs/Fee_Structure_PG_2018-19_admitted.pdf | Page: 1 | Title: Fee structure for M.Tech./M.Plan.
  [5] Source: ../pdfs/FeeUG2025-26.pdf | Page: 4 | Title: Fee Structure for B. Tech.
  [6] Source: ../pdfs/Fee_Structure_PG_2018-19_admitted.pdf | Page: 0 | Title: Fee structure for M.Tech./M.Plan.
  [7] Source: ../pdfs/Fee_Structure_UG_2017-18.pdf | Page: 0 | Title: Fee Structure for B. Tech. /B.Arch.
  [8] Source: ../pdfs/Final_Fee_Structure_PG_20

[Document(id='5cec31ff-209e-4db2-8492-8cd5b1a2ff75', metadata={'moddate': '2017-04-27T18:21:24+05:30', 'producer': 'Microsoft® Word 2013', 'Dept': 'Institute', 'total_pages': 3, 'title': 'Fee Structure for B. Tech. /B.Arch.', 'creator': 'Microsoft® Word 2013', 'page': 0, 'doc_type': 'Fee Structure', 'audience': 'UG', 'summary': 'This document details the fee structure for B.Tech and B.Arch students admitted in the 2016-17 session.', 'source': '../pdfs/Fee_Structure_UG_2016-17.pdf', 'author': 'IBM', 'year': '2016', 'page_label': '1', 'creationdate': '2017-04-27T18:21:24+05:30'}, page_content='MALAVIYA NATIONAL INSTITUTE OF TECHNOLOGY JAIPUR \n \n \n Fee Structure for B. Tech. /B.Arch. students admitted in the session 2016-17 \n \nTUITION FEE \n \nS. No. \nHead of Fee \n \nOdd Semester & Even Semester \nOP/OBC PH/SC/ST \nIncome  \nBelow 1 Lac \nIncome                                  \n1 Lac to 5 Lac \nIncome                   \nAbove 5 Lac All \n1. Tuition Fee per Semester 0 20,834.00 6

In [ ]:
# Test 4: Faculty Query
retriever.retrieve("Quantum computing classes for faculty")

--- 1. Wide Retrieval for: 'Quantum computing classes for faculty' ---
--- 2. Reranking 10 documents ---
--- 2. Reranking 10 documents ---

--- Top Selected Documents ---
Score: 1.0 | Doc Source: ../pdfs/QT-07_Quantum_Sensing_Brochure_upd.pdf
Reasoning: This document is a notice for an 'Online Faculty Programme on Quantum Sensing' which is part of an 'AICTE Approved Minor Course Curriculum on Quantum Computing' for the year 2025, explicitly targeting faculty. This directly matches all user constraints.
Score: 1.0 | Doc Source: ../pdfs/QT-03_BASIC_QUANTUM_PROGRAMMING_2025.pdf
Reasoning: This document is a notice for an 'Online Faculty Programme on Basic Quantum Programming' which is part of an 'AICTE Approved Minor Course Curriculum on Quantum Computing' for the year 2025, explicitly targeting faculty. This directly matches all user constraints.
Score: 1.0 | Doc Source: ../pdfs/QT-05_Quantum_Computation_Brochure_updated_1.pdf
Reasoning: This document is a notice for an 'Online Faculty P

[Document(id='168d713b-bef6-4d56-82f5-8998a0eda91a', metadata={'total_pages': 1, 'author': 'x', 'summary': 'An intensive 20-day online training programme on Quantum Sensing is being organized for faculty and doctoral students.', 'moddate': '2025-09-17T06:16:12+00:00', 'title': 'AICTE Approved Minor Course Curriculum on Quantum Computing', 'creator': 'Microsoft® Word 2016', 'doc_type': 'Notice', 'creationdate': '2025-09-17T06:16:12+00:00', 'page': 0, 'audience': 'Faculty', 'Dept': 'Institute', 'producer': 'www.ilovepdf.com', 'year': '2025', 'source': '../pdfs/QT-07_Quantum_Sensing_Brochure_upd.pdf', 'page_label': '1'}, page_content='AICTE Approved Minor Course Curriculum \non Quantum Computing \n                  \n  \n \nIIT Kanpur, IIT Roorkee, IIT Guwahati,     \nNIT Patna, NIT Warangal, IIITDM Jabalpur \nOnline Faculty Programme on  \nQT – 07 :  \nQuantum Sensing   \nSept 26 – Oct 17, 2025 \nTwenty Days (Mon to Sat) \nTime: 2 – 4 PM (Daily 2 Hours) \n \nAn intensive 20-day-40-hour T

In [75]:
# Test 5: Policy
retriever.retrieve("What is the unfair means policy policy or UFM of the instituion")

--- 1. Wide Retrieval for: 'What is the unfair means policy policy or UFM of the instituion' ---

[Log] Initial 10 Documents Retrieved:
  [0] Source: ../pdfs/Sem_promotion_policy_1st_year.pdf | Page: 0 | Title: Minimum Requirement to continue in the program & Promotion
  [1] Source: ../pdfs/sou_motu_n.pdf | Page: 7 | Title: Organisation and Function
  [2] Source: ../pdfs/Semester_Promotion_Policy_2015.pdf | Page: 0 | Title: Promotion of B.Tech/B.Arch students
  [3] Source: ../pdfs/Notice_eligible_ineligible_applicants_for_the_post_Rregistrar.pdf | Page: 1 | Title: Final List of Eligible/ Ineligible Applicants for the Post of Registrar
  [4] Source: ../pdfs/Sem_promotion_policy_1st_year.pdf | Page: 4 | Title: Minimum Requirement to continue in the program & Promotion
  [5] Source: ../pdfs/Sem_promotion_policy_3sem.pdf | Page: 0 | Title: Minimum Requirement for B.Tech/B.Arch Continuation & Promotion
  [6] Source: ../pdfs/PM_VidyalaxmiSchemes.pdf | Page: 1 | Title: Implementation of PM-Vi

[Document(id='eb6ab811-f286-4ee2-b0b7-53961f48adc2', metadata={'page_label': '5', 'page': 4, 'source': '../pdfs/Sem_promotion_policy_1st_year.pdf', 'title': 'Minimum Requirement to continue in the program & Promotion', 'doc_type': 'Policy', 'audience': 'UG', 'year': 'Unknown', 'summary': 'This document outlines the policy regarding minimum academic performance requirements for B.Tech./B.Arch. students to progress to subsequent semesters at MNIT Jaipur.', 'creator': 'Microsoft® Office Word 2007', 'total_pages': 9, 'Dept': 'Institute', 'moddate': '2015-04-23T17:12:10+05:30', 'producer': 'Microsoft® Office Word 2007', 'author': 'lavab', 'creationdate': '2015-04-23T17:12:10+05:30'}, page_content='Students who have failed in one semester / taken semester withdrawal / rusticated fo r one \nsemester / not promoted to higher semester on account of N -4 rule or any other reason \n/Academically deficient student  not able to register for higher semester courses due to \nregistration of pending  